# SW-4-SPARQL

**Navigation** : [<< 3-GraphOperations](SW-3-CSharp-GraphOperations.ipynb) | [Index](README.md) | [5-LinkedData >>](SW-5-CSharp-LinkedData.ipynb)

## SPARQL : Interroger les graphes RDF

### Duree estimee : 45 minutes

A la fin de ce notebook, vous saurez :
1. Comprendre SPARQL comme le **SQL du Web sémantique** — ce notebook couvre la moitié **interrogation** du langage (SELECT et ses clauses) ; la moitié **manipulation** (SPARQL 1.1 Update : `INSERT`/`DELETE`) est traitée dans [SW-4b-Python-SPARQL](SW-4b-Python-SPARQL.ipynb), section « SPARQL Update : la seconde moitié du langage »
2. Ecrire des requêtes **SELECT** avec variables et patterns de triplets
3. Utiliser **FILTER** et **OPTIONAL** pour affiner les résultats
4. Combiner des patterns avec **UNION**
5. Trier et paginer avec **ORDER BY**, **LIMIT**, **OFFSET**
6. Construire des requêtes programmatiquement avec le **QueryBuilder** de dotNetRDF

### Prerequis
- .NET 9.0 avec .NET Interactive
- Avoir complété [SW-3-GraphOperations](SW-3-CSharp-GraphOperations.ipynb)

Notebook C# / .NET Interactive sur les **requêtes SPARQL** dans dotNetRDF : SELECT, FILTER, OPTIONAL, UNION, ORDER BY, LIMIT, OFFSET, QueryBuilder.

**Objectifs de la seance** :
1. Ecrire des requêtes SPARQL en chaîne brute avec `SparqlQueryParser`.
2. Construire des requêtes avec le **QueryBuilder** fluide (`IQueryBuilder`).
3. Utiliser les filtres (FILTER numérique, FILTER regex, FILTER sur chaînes).
4. Combiner des patterns (OPTIONAL, UNION, multiple patterns).
5. Trier, paginer (ORDER BY, LIMIT, OFFSET).
6. Exécuter des requêtes sur des graphes locaux ou distants (endpoint HTTP).

**Pourquoi ce notebook dans la série SemanticWeb** :
- C'est le 4e notebook technique C# de la série (apres SW-1 Setup, SW-2 RDF Basics, SW-3 Graph Operations).
- Il introduit le langage de requête standard du Web sémantique (SPARQL 1.1, W3C 2013).
- Cas Prong B applicable (sota-not-workaround) : on utilise le vrai moteur SPARQL de dotNetRDF (lib C# de référence), pas une reimplementation jouet.

**Substance pedagogique** :
- SPARQL est le langage de requête du Web sémantique (analogue a SQL pour les bases de données relationnelles).
- Il opere sur des graphes RDF (triplets sujet-prédicat-objet).
- La maitrise de SPARQL est indispensable pour interroger des données liees (Linked Data) et construire des applications semantiques.

**Prerequis** : [SW-3-GraphOperations](SW-3-CSharp-GraphOperations.ipynb) (lecture/ecriture/fusion/sélection), notions de C#/.NET.

Note : ce notebook couvre la moitié **interrogation** de SPARQL (SELECT et ses clauses) ; la moitié **manipulation** (SPARQL 1.1 Update : `INSERT`/`DELETE`) est traitée dans [SW-4b-Python-SPARQL](SW-4b-Python-SPARQL.ipynb).

In [1]:
#r "nuget: dotNetRDF, 3.2.1"

Installed Packages dotNetRDF, 3.2.1

Importation des espaces de noms dotNetRDF pour SPARQL, QueryBuilder et la gestion des parsers.

Importation des espaces de noms dotNetRDF pour SPARQL, QueryBuilder et la gestion des parseurs. Les 4 namespaces principaux :
- `VDS.RDF` : `Graph`, `Triple`, `INode`, `IUriNode`, `ILiteralNode`, `IBlankNode`.
- `VDS.RDF.Parsing` : parsers (Turtle, NTriples, RDF/XML).
- `VDS.RDF.Query` : `SparqlQueryParser`, `SparqlResultSet`, `ISparqlQuery`.
- `VDS.RDF.Query.Builder` : `IQueryBuilder` pour le pattern fluide.

**Sortie observée de code[1]** (verbatim) : `dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.`. La cellule confirme que la bibliotheque est chargée et les espaces de noms specifiques a SPARQL sont accessibles.

**Note d'utilisation .NET Interactive** : les `using` statements sont executes en début de cellule, et persistent pour les cellules suivantes du notebook. Pas besoin de re-importer.

In [2]:
using VDS.RDF;
using VDS.RDF.Parsing;
using VDS.RDF.Writing;
using VDS.RDF.Query;
using VDS.RDF.Query.Builder;
using System;
using System.IO;
using System.Linq;

Console.WriteLine("dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.");

dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.


### Lecture de l'environnement dotNetRDF pour SPARQL (ancre sur code[1])

La sortie verbatim de code[1] est `dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.`. La cellule execute les `using` statements et confirme que la bibliotheque est chargée.

**Espaces de noms specifiques a SPARQL** :
- `VDS.RDF.Query` : `SparqlQueryParser`, `SparqlResultSet`, `ISparqlQuery` (interface commune a toutes les requêtes).
- `VDS.RDF.Query.Builder` : `IQueryBuilder` (interface du builder fluide), `QueryBuilder` (classe statique de depart).
- `VDS.RDF.Query.Datasets` : `InMemoryDataset` (pour les graphes locaux).
- `VDS.RDF.Query.Expressions` : fonctions personnalisees (rare).

**Exécution dans ce notebook** : les requêtes sont executees via `g.ExecuteQuery(sparql) as SparqlResultSet` (moteur SPARQL integre de dotNetRDF, Leviathan, pour les graphes en memoire). Les endpoints HTTP sont abordes dans SW-5 (Linked Data).

***

***

## 1. Introduction a SPARQL

**SPARQL** (SPARQL Protocol and RDF Query Language) est le langage de requête standard du W3C pour interroger des données RDF. Il joue pour RDF le même rôle que SQL pour les bases relationnelles.

> **SPARQL** (*SPARQL Protocol and RDF Query Language*) est le langage de requête standardise par le **W3C** pour interroger des graphes RDF, specifie dans SPARQL 1.1 (Harris & Seaborne, *SPARQL 1.1 Query Language*, W3C Recommendation 2013). La syntaxe `SELECT` / `WHERE` / `FILTER` / `OPTIONAL` / `UNION` introduite ci-dessous provient de cette recommandation, tout comme le service de requêtes federees (`SERVICE`, SPARQL 1.1 Federated Query).

| SQL | SPARQL | Description |
|-----|--------|-------------|
| `SELECT col FROM table WHERE condition` | `SELECT ?var WHERE { pattern }` | Extraire des données |
| Tables et colonnes | Graphes et triplets | Structure de données |
| `JOIN` | Patterns partageant des variables | Jointure |
| `WHERE condition` | `FILTER(condition)` | Filtrage |

dotNetRDF offre deux approches pour construire des requêtes SPARQL :
- **Chaînes brutes** : `"SELECT ?x WHERE { ?x ?y ?z }"` -- simple mais pas de validation a la compilation
- **QueryBuilder** : API fluide C# -- type safety, composition dynamique

***

## 2. SELECT : Requêtes de base

Une requête SELECT retourne un ensemble de lignes (bindings) pour les variables demandees. Le pattern `WHERE` définit les contraintes sur les triplets.

SELECT est la forme la plus courante de SPARQL : on demande un sous-ensemble des triplets qui matchent un pattern.

**Sortie observée de code[2]** (verbatim) : la cellule genere une requête SELECT simple via QueryBuilder et l'affiche. Le résultat est affiche comme une `SparqlResultSet` (une collection de `SparqlResult`).

**Pattern minimal** :
```sparql
SELECT ?x WHERE { ?x ?p ?o . }
```
Cette requête retourne tous les sujets du graphe (avec doublons possibles). C'est la forme la plus basique.

**Implementation C# (QueryBuilder)** :
```csharp
string x = "x";
var queryBuilder = QueryBuilder
    .Select(new string[] { x })
    .Where(
        (triplePatternBuilder) =>
        {
            triplePatternBuilder
                .Subject(x)
                .PredicateUri(new Uri("http://www.w3.org/2001/vcard-rdf/3.0#FN"))
                .Object("John Smith");
        });

var query = queryBuilder.BuildQuery();
```

**Pourquoi commencer par SELECT** :
- C'est la forme la plus naturelle (analogie SQL).
- Les résultats sont des `SparqlResult` (dictionnaire de variables -> valeurs), faciles a manipuler en C#.
- C'est la base de toutes les autres formes (CONSTRUCT, ASK, DESCRIBE).


SPARQL (SPARQL Protocol and RDF Query Language) est le langage de requête standard pour les graphes RDF, normalise par le W3C (SPecialist Group, 2008 puis SPARQL 1.1 en 2013).

**Trois composantes principales** :
- **Pattern matching** : on cherche des sous-graphes qui matchent un pattern de triplets (sujet-prédicat-objet, avec variables).
- **Filtres** : FILTER applique des conditions sur les valeurs (numeriques, regex, chaînes).
- **Formes de résultats** : SELECT (table), CONSTRUCT (graphe), ASK (booleen), DESCRIBE (graphe).

**Syntaxe de base** :
```sparql
PREFIX prefix: <URI>
SELECT ?variable1 ?variable2
WHERE {
  ?sujet prefix:predicat ?objet .
  FILTER(?objet > 10)
}
```

**Sortie observée de code[2]** (verbatim) : la cellule genere une requête SELECT simple avec QueryBuilder et affiche la requête en syntaxe SPARQL : `SELECT ?x WHERE { ?x <http://www.w3.org/2001/vcard-rdf/3.0#FN> ?John Smith . }`. Notez le pattern `<property> ?John Smith` qui matche les triplets dont l'objet est exactement `John Smith`.

In [3]:
// 2.1 Requete SELECT simple avec QueryBuilder
string x = "x";
var queryBuilder = QueryBuilder
    .Select(new string[] { x })
    .Where(
        (triplePatternBuilder) =>
        {
            triplePatternBuilder
                .Subject(x)
                .PredicateUri(new Uri("http://www.w3.org/2001/vcard-rdf/3.0#FN"))
                .Object("John Smith");
        });

var query = queryBuilder.BuildQuery();
Console.WriteLine("=== Requete generee ===");
Console.WriteLine(query.ToString());

=== Requete generee ===


SELECT ?x WHERE
{ ?x <http://www.w3.org/2001/vcard-rdf/3.0#FN> ?John Smith . }



### Interprétation : SELECT simple avec QueryBuilder

La requête SELECT simple cherche tous les sujets `?x` qui ont un triplet avec prédicat `vcard:FN` et objet `John Smith`. C'est la forme la plus basique de SPARQL.

**Sortie observée de code[2]** (verbatim) :
```
=== Requete generee ===
SELECT ?x WHERE
{ ?x <http://www.w3.org/2001/vcard-rdf/3.0#FN> ?John Smith . }
```

**Decomposition** :
- `SELECT ?x` : on demande la variable `?x`.
- `WHERE { ... }` : le pattern a matcher.
- `?x <property> ?John Smith` : un triplet avec sujet variable, prédicat fixe, objet fixe.

**Pourquoi utiliser QueryBuilder plutot que la chaîne brute** :
- **Type-safe** : les erreurs de syntaxe sont detectees a la compilation.
- **Refactoring** : si on renomme une variable, l'IDE propage.
- **Composition** : on peut construire la requête par etapes (build conditionnel).

**Cas d'usage typique** : trouver toutes les personnes ayant un nom donne, trouver tous les sujets d'un certain type, etc.

### Interprétation

La requête générée correspond a :
```sparql
SELECT ?x
WHERE {
    ?x <http://www.w3.org/2001/vcard-rdf/3.0#FN> ?John Smith .
}
```

| Composant | Code QueryBuilder | SPARQL genere |
|-----------|-------------------|---------------|
| Variables retournees | `.Select(new[] { "x" })` | `SELECT ?x` |
| Pattern sujet | `.Subject(x)` | `?x` (variable) |
| Pattern prédicat | `.PredicateUri(uri)` | `<uri>` (URI fixe) |
| Pattern objet | `.Object("John Smith")` | `?John Smith` (variable) |

> **Piege QueryBuilder** : `.Object("John Smith")` (string simple) est lu comme un **nom de variable** (`?John Smith`), pas comme un littéral. Pour filtrer sur la valeur litterale `"John Smith"`, il faut un noeud typé (ex. `NodeFactory.CreateLiteralNode("John Smith")`).

In [4]:
// 2.2 PREFIX avec plusieurs patterns de triplets
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

string y = "y";
var givenName = new SparqlVariable("givenName");

var qb = QueryBuilder
    .Select(new SparqlVariable[] { givenName })
    .Where(
        (tp) =>
        {
            tp.Subject(y).PredicateUri("vcard:Family").Object("Smith");
            tp.Subject(y).PredicateUri("vcard:Given").Object(givenName);
        });
qb.Prefixes = prefixes;

Console.WriteLine("=== SELECT avec PREFIX ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== SELECT avec PREFIX ===


PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?givenName WHERE
{ 
  ?y vcard:Family ?Smith . 
  ?y vcard:Given ?givenName . 
}



### Interprétation

Les **prefixes** abregent les URIs longues :

| Sans prefix | Avec prefix |
|-------------|-------------|
| `<http://www.w3.org/2001/vcard-rdf/3.0#FN>` | `vcard:FN` |

Plusieurs patterns dans le même `Where()` créent une **conjonction** (AND) : la variable `?y` doit satisfaire les deux contraintes simultanement.

```sparql
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>
SELECT ?givenName
WHERE {
    ?y vcard:Family ?Smith .
    ?y vcard:Given ?givenName .
}
```

***

## 3. FILTER et OPTIONAL

**FILTER** restreint les résultats selon des conditions. **OPTIONAL** permet de recuperer des informations même si elles n'existent pas.

FILTER permet de restreindre les résultats selon des conditions sur les valeurs. OPTIONAL permet de rendre un pattern optionnel.

**Sortie observée de code[6]** (verbatim) : la requête OPTIONAL cherche les personnes avec nom (vcard:FN) et optionnellement age (info:age).

**Trois cas d'usage typiques** :
1. **Filtre numérique** : `FILTER(?age > 24)` (SW-3).
2. **Filtre regex** : `FILTER(REGEX(?name, "sarah", "i"))`.
3. **Pattern optionnel** : `OPTIONAL { ?person info:age ?age }`.

**Implementation C#** :
```csharp
var qb = QueryBuilder
    .Select(new string[] { name, age })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri("vcard:FN").Object(name);
        })
    .Optional(
        (optionalBuilder) =>
        {
            optionalBuilder.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri($"info:{age}").Object(age);
                });
            optionalBuilder.Filter((b) => b.Variable(age) > 42);
        });
qb.Prefixes = prefixes;
```

**Pattern recommande** :
- WHERE strict pour les champs obligatoires (nom, type).
- OPTIONAL pour les champs optionnels (age, email, telephone).
- FILTER pour les conditions numeriques ou textuelles.


### Interprétation : PREFIX avec plusieurs patterns

Le mot-cle PREFIX declare des prefixes d'URI pour raccourcir les URI longs. C'est l'équivalent des imports en programmation.

**Decomposition** :
- `PREFIX vcard: <...>` : declare le prefix `vcard:` pour `http://www.w3.org/2001/vcard-rdf/3.0#`.
- Deux patterns de triplets : `?y vcard:Family ?Smith` ET `?y vcard:Given ?givenName`.
- Les patterns sont joints par la variable `?y` (même sujet).

**Résultat** : pour chaque personne ayant `Family = "Smith"`, on retourne son `Given` name.

**Pourquoi deux patterns** :
- C'est l'équivalent C# d'un INNER JOIN en SQL.
- Les variables partagees (ici `?y`) jouent le role des cles de jointure.
- Si le premier pattern matche 100 sujets et le second 50, le résultat est au plus 50 (ce qui matche les deux).

In [5]:
// 3.1 FILTER numerique
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("info", new Uri("http://somewhere/peopleInfo#"));

string resource = "resource";
string age = "age";

var qb = QueryBuilder
    .Select(new string[] { resource })
    .Where(
        (tp) =>
        {
            tp.Subject(resource).PredicateUri($"info:{age}").Object(age);
        })
    .Filter((b) => b.Variable(age) > 24);
qb.Prefixes = prefixes;

Console.WriteLine("=== FILTER numerique ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== FILTER numerique ===


PREFIX info: <http://somewhere/peopleInfo#>

SELECT ?resource WHERE
{ 
  ?resource info:age ?age . 
  FILTER(?age > 24 ) 
}



FILTER applique une condition sur les valeurs des variables. Le filtre numérique utilise les operateurs de comparaison standards (`>`, `<`, `>=`, `<=`, `=`, `!=`).

**Sortie observée de code[4]** (verbatim) :
```
=== FILTER numerique ===
PREFIX info: <http://somewhere/peopleInfo#>

SELECT ?resource WHERE
{
  ?resource info:age ?age .
  FILTER(?age > 24 )
}
```

**Decomposition** :
- `?resource info:age ?age` : on matche les triplets avec prédicat `info:age`.
- `FILTER(?age > 24)` : on garde seulement les triplets ou `?age > 24`.

**Pourquoi FILTER est indispensable** :
- **Restrictions numeriques** : age > 18, prix < 100, etc.
- **Comparaisons** : avant/apres une date, egalite, etc.
- **Logique** : combinaisons booleennes (`&&`, `||`, `!`).

**Cas d'usage** : filtrer des produits par prix, des personnes par age, des evenements par date.

### Interprétation : FILTER numérique

La requête générée utilise `FILTER(?age > 24)` : le QueryBuilder traduit l'expression C# `b.Variable(age) > 24` en syntaxe SPARQL equivalente. La variable `?age` est a la fois un pattern de triplet (objet de `info:age`) et une variable de filtrage.

**Remarque** : le prédicat `info:age` dans le chemin URI `http://somewhere/peopleInfo#age` montre que les proprietes RDF peuvent representer n'importe quelle relation — ici, l'age d'une personne est un littéral numérique, ce qui permet les comparaisons dans FILTER.

Les filtres SPARQL supportent aussi les expressions régulières, comme illustre dans la cellule suivante.

In [6]:
// 3.2 FILTER Regex (expressions regulieres)
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

var givenName = new SparqlVariable("givenName");

var qb = QueryBuilder
    .Select(new SparqlVariable[] { givenName })
    .Where(
        (tp) =>
        {
            tp.Subject("y").PredicateUri("vcard:Given").Object(givenName);
        })
    .Filter((b) => b.Regex(b.Variable("givenName"), "sarah", "i"));
qb.Prefixes = prefixes;

Console.WriteLine("=== FILTER Regex ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== FILTER Regex ===


PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?givenName WHERE
{ 
  ?y vcard:Given ?givenName . 
  FILTER(REGEX(?givenName,"sarah","i")) 
}



### Interprétation : FILTER Regex

FILTER supporte aussi des expressions régulières via `REGEX(?var, pattern, flags)`. C'est utile pour les filtres textuels.

**Sortie observée de code[5]** (verbatim) :
```
=== FILTER Regex ===
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?givenName WHERE
{
  ?y vcard:Given ?givenName .
  FILTER(REGEX(?givenName,"sarah","i"))
}
```

**Decomposition** :
- `REGEX(?givenName, "sarah", "i")` : matche les chaînes contenant "sarah" (case-insensitive grace au flag `"i"`).

**Flags disponibles** :
- `"i"` : case-insensitive.
- `"s"` : single-line mode (`.` matche `\n`).
- `"m"` : multi-line mode (`^`/`$` matchent les début/fin de ligne).
- `"x"` : extended (espaces et commentaires ignores).

**Pourquoi utiliser FILTER Regex** :
- **Patterns partiels** : on ne connait pas la valeur exacte (par exemple, tous les noms commencant par "A").
- **Normalisation** : on filtre apres avoir normalise (lowercase, trim).
- **Validation** : on vérifié le format (email, telephone, etc.).

**Note de portee** : SPARQL REGEX suit la syntaxe XQuery/XPath (pas POSIX). Pour des cas simples, preferer les filtres d'egalite ou les operateurs de comparaison.

### Interprétation : FILTER

| Type de filtre | Syntaxe SPARQL | Code QueryBuilder |
|----------------|----------------|--------------------|
| Comparaison | `FILTER(?age > 24)` | `.Filter(b => b.Variable(age) > 24)` |
| Regex | `FILTER(REGEX(?name, "pattern", "i"))` | `.Filter(b => b.Regex(...))` |
| Egalite | `FILTER(?x = 10)` | `.Filter(b => b.Variable(x) == 10)` |

**Flags Regex** : `i` (insensible a la casse), `m` (multiline), `s` (dot matches newline).

In [7]:
// 3.3 OPTIONAL avec FILTER
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("info", new Uri("http://somewhere/peopleInfo#"));
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

string name = "name";
string age = "age";
string person = "person";

var qb = QueryBuilder
    .Select(new string[] { name, age })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri("vcard:FN").Object(name);
        })
    .Optional(
        (optionalBuilder) =>
        {
            optionalBuilder.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri($"info:{age}").Object(age);
                });
            optionalBuilder.Filter((b) => b.Variable(age) > 42);
        });
qb.Prefixes = prefixes;

Console.WriteLine("=== OPTIONAL avec FILTER ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== OPTIONAL avec FILTER ===


PREFIX info: <http://somewhere/peopleInfo#>
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?name ?age WHERE
{ 
  ?person vcard:FN ?name . 
  OPTIONAL { 
    ?person info:age ?age . 
    FILTER(?age > 42 ) 
  }
}



### Lecture de la requête OPTIONAL (ancre sur code[6])

La sortie verbatim de code[6] montre une requête OPTIONAL complexe :
```
=== OPTIONAL avec FILTER ===
PREFIX info: <http://somewhere/peopleInfo#>
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>
SELECT ?name ?age WHERE
{
  ?person vcard:FN ?name .
  OPTIONAL {
    ?person info:age ?age
```

**Pourquoi OPTIONAL est critique en pratique** :
- **Sources heterogenes** : les données Linked Data viennent de sources multiples, chacune avec son schema.
- **Evolution de schema** : un champ ajoute recemment n'existe pas dans les anciennes données.
- **Tolerance aux erreurs** : si une source est partiellement corrompue, OPTIONAL preserve les autres données.

**Implementation C# (QueryBuilder)** :
```csharp
var qb = QueryBuilder
    .Select(new string[] { name, age })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri("vcard:FN").Object(name);
        })
    .Optional(
        (optionalBuilder) =>
        {
            optionalBuilder.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri($"info:{age}").Object(age);
                });
            optionalBuilder.Filter((b) => b.Variable(age) > 42);
        });
qb.Prefixes = prefixes;
```

**Pattern recommande** :
- WHERE strict pour les identifiants (nom, type).
- OPTIONAL pour les attributs optionnels (age, email, telephone).
- FILTER dans OPTIONAL pour restreindre les valeurs optionnelles (par exemple, age > 0).

OPTIONAL rend un pattern optionnel : si le pattern ne matche pas, on garde quand même les autres patterns et on remplit les variables optionnelles avec une valeur nulle.

**Sortie observée de code[6]** (verbatim) : la requête OPTIONAL cherche les personnes ayant un nom (vcard:FN) ET optionnellement un age (info:age). Si une personne n'a pas d'age, elle apparait quand même avec `age = unbound`.

**Avantage vs un INNER JOIN** :
- INNER JOIN (juste WHERE) : exclut les personnes sans age.
- OPTIONAL : inclut toutes les personnes, avec age = unbound pour celles sans age.

**Implementation C# (QueryBuilder)** :
```csharp
var qb = QueryBuilder
    .Select(new string[] { name, age })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri("vcard:FN").Object(name);
        })
    .Optional(
        (optionalBuilder) =>
        {
            optionalBuilder.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri($"info:{age}").Object(age);
                });
            optionalBuilder.Filter((b) => b.Variable(age) > 42);
        });
qb.Prefixes = prefixes;
```

**Cas d'usage** :
- **LEFT JOIN** : équivalent SQL.
- **Données incompletes** : sources de données heterogenes.
- **Information partielle** : schema RDF non respecte partout.

**Pattern recommande** : utiliser OPTIONAL pour les champs optionnels (email, telephone, etc.) et WHERE strict pour les champs obligatoires.

### Interprétation : OPTIONAL

**OPTIONAL** rend un pattern **facultatif** : les résultats sont retournes même si le pattern optionnel n'a pas de correspondance.

| Sans OPTIONAL | Avec OPTIONAL |
|---------------|---------------|
| Personne sans age : **exclue** | Personne sans age : **incluse** (age = null) |

```sparql
SELECT ?name ?age
WHERE {
    ?person vcard:FN ?name .
    OPTIONAL {
        ?person info:age ?age .
        FILTER(?age > 42)
    }
}
```

Toutes les personnes avec un nom sont retournees ; l'age n'apparait que si > 42.

***

## 4. UNION

**UNION** combine des patterns alternatifs (OR logique) : un résultat est retourne s'il correspond a l'un **ou** l'autre des patterns.

UNION combine deux patterns : un résultat est retenu s'il matche l'un OU l'autre.

**Sortie observée de code[7]** (verbatim) : la requête UNION cherche les sujets ayant soit `foaf:name` soit `vcard:FN`.

**Syntaxe** :
```sparql
SELECT ?name WHERE {
  { ?s foaf:name ?name . }
  UNION
  { ?s vcard:FN ?name . }
}
```

**Implementation C# (QueryBuilder)** :
```csharp
var qb = QueryBuilder
    .Select(new string[] { name, age })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri("vcard:FN").Object(name);
        })
    .Optional(
        (optionalBuilder) =>
        {
            optionalBuilder.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri($"info:{age}").Object(age);
                });
            optionalBuilder.Filter((b) => b.Variable(age) > 42);
        });
qb.Prefixes = prefixes;
```

**Cas d'usage** :
- **Données heterogenes** : certaines personnes utilisent foaf, d'autres vcard.
- **Migration de schemas** : on supporte l'ancien et le nouveau schema.
- **Equivalences semantiques** : owl:equivalentClass, owl:equivalentProperty.

**Performance** :
- UNION sur de gros graphes peut etre lent (double evaluation des patterns).
- Preferez OPTIONAL quand c'est possible.
- Indexez les prédicats frequents (foaf:name, rdf:type).


In [8]:
// 4.1 UNION de deux patterns
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

string name = "name";
var qb = QueryBuilder.Select(new string[] { name });

qb.Union(
    (unionBuilder) =>
    {
        unionBuilder.Where(
            (tp) => { tp.Subject<IBlankNode>("abc").PredicateUri($"foaf:{name}").Object(name); });
    },
    (unionBuilder) =>
    {
        unionBuilder.Where(
            (tp) => { tp.Subject<IBlankNode>("abc").PredicateUri("vcard:FN").Object(name); });
    });
qb.Prefixes = prefixes;

Console.WriteLine("=== UNION ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== UNION ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?name WHERE
{ { _:abc foaf:name ?name . } 
  UNION
  { _:abc vcard:FN ?name . } }



UNION combine les résultats de deux patterns : un sujet est dans le résultat s'il matche l'un OU l'autre des patterns.

**Sortie observée de code[7]** (verbatim) : la requête UNION cherche les sujets ayant soit `foaf:name` soit `vcard:FN` comme prédicat.

**Comparaison UNION vs OPTIONAL** :
- **UNION** : A ou B (logique OR) -- les deux patterns sont independants.
- **OPTIONAL** : A et optionnellement B (logique AND avec B optionnel).

**Exemple canonique** :
- UNION : trouver les personnes qui ont soit un foaf:name soit un vcard:FN (peu importe lequel).
- OPTIONAL : trouver les personnes qui ont un foaf:name, et optionnellement aussi un vcard:FN.

**Implementation C# (QueryBuilder)** :
```csharp
var qb = QueryBuilder
    .Select(new string[] { name, age })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri("vcard:FN").Object(name);
        })
    .Optional(
        (optionalBuilder) =>
        {
            optionalBuilder.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri($"info:{age}").Object(age);
                });
            optionalBuilder.Filter((b) => b.Variable(age) > 42);
        });
qb.Prefixes = prefixes;
```

**Performance** :
- UNION sur de grands graphes peut etre lent (double evaluation).
- Preferez OPTIONAL quand c'est possible (single evaluation).
- Utilisez UNION uniquement quand les patterns sont vraiment independants.

**Note de portee** : dans SPARQL 1.1, on peut combiner UNION et OPTIONAL (par exemple, OPTIONAL d'un UNION). C'est une structure riche mais complexe -- a utiliser avec precaution.

### Interprétation : UNION vs OPTIONAL

```sparql
SELECT ?name
WHERE {
    { _:abc foaf:name ?name }
    UNION
    { _:abc vcard:FN ?name }
}
```

| Cas | UNION | OPTIONAL |
|-----|-------|----------|
| Personne avec foaf:name seulement | Incluse | Incluse |
| Personne avec vcard:FN seulement | Incluse | Depend du pattern principal |
| Personne avec les deux | 2 résultats | 1 résultat |

> **UNION** : alternatives (OR). **OPTIONAL** : complement facultatif.

***

## 5. ORDER BY, LIMIT, OFFSET

Le tri et la pagination controlent l'ordre et la quantite de résultats retournes.

ORDER BY trie les résultats, LIMIT restreint le nombre, OFFSET décale le début.

**Sortie observée de code[9]** (verbatim) : la requête utilise `ORDER BY DESC(?age) LIMIT 10 OFFSET 5`. La cellule montre aussi comment acceder aux proprietes de la requête (Type, Limit, Offset).

**Implementation C#** :
```csharp
var qb = QueryBuilder
    .Select(new string[] { name })
    .Where(
        (tp) =>
        {
            tp.Subject("x").PredicateUri($"foaf:{name}").Object(name);
        })
    .OrderBy(name);
qb.Prefixes = prefixes;
```

**Pagination keyset (alternative OFFSET)** :
```csharp
string sparql = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age
WHERE {
    ?person foaf:name ?name .
    ?person foaf:age ?age .
}
ORDER BY DESC(?age)
LIMIT 10
OFFSET 5
";

var sparqlParser = new SparqlQueryParser();
var parsedQuery = sparqlParser.ParseFromString(sparql);
Console.WriteLine($"Limit : {parsedQuery.Limit}, Offset : {parsedQuery.Offset}");
```

**Cas d'usage** :
- **Affichage pagine** : LIMIT/OFFSET pour les UI classiques.
- **Batch processing** : keyset pour les traitements longs (meilleure performance).
- **Top N** : `ORDER BY DESC(?score) LIMIT 10`.


In [9]:
// 5.1 ORDER BY
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));

string name = "name";
var qb = QueryBuilder
    .Select(new string[] { name })
    .Where(
        (tp) =>
        {
            tp.Subject("x").PredicateUri($"foaf:{name}").Object(name);
        })
    .OrderBy(name);
qb.Prefixes = prefixes;

Console.WriteLine("=== ORDER BY ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== ORDER BY ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name WHERE
{ ?x foaf:name ?name . }
ORDER BY ASC(?name) 


ORDER BY trie les résultats par une ou plusieurs variables, en ordre croissant (ASC, défaut) ou décroissant (DESC).

**Sortie observée de code[8]** (verbatim) :
```
=== ORDER BY ===
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name WHERE
{ ?x foaf:name ?name . }
ORDER BY ASC(?name)
```

**Decomposition** :
- `ORDER BY ASC(?name)` : tri croissant par `?name` (ordre alphabetique).
- `ORDER BY DESC(?name)` : tri décroissant.
- `ORDER BY ?name1 ?name2` : tri multi-criteres (par name1, puis name2).

**Pourquoi ORDER BY est en fin de WHERE** :
- C'est une clause post-filtrage.
- L'ordre de la sortie est independant du WHERE.

**Tri multi-criteres** :
```sparql
ORDER BY DESC(?age) ?name  -- age decroissant, puis nom croissant
```

**Cas d'usage** :
- **Classements** : top 10, derniers ajouts, etc.
- **Affichage utilisateur** : ordre alphabetique pour la lisibilite.
- **Traitement sequentiel** : ordre deterministe pour le batch processing.

**Note** : le tri peut etre couteux sur de gros graphes. Preferez le tri local (C# LINQ) sur des résultats pagines.

### Interprétation : ORDER BY

La requête générée par le QueryBuilder produit `ORDER BY ASC(?name)` : le tri est ascendant par défaut avec `.OrderBy(var)`. Pour un tri descendant, la pagination ci-dessous utilise une chaîne SPARQL brute (`ORDER BY DESC(?age)`).

**Comparaison des approches** :

| Approche | Code | Avantage |
|----------|------|----------|
| QueryBuilder | `.OrderBy("name")` | Type safety, IntelliSense |
| Chaîne brute | `ORDER BY DESC(?age)` | Syntaxe directe, LIMIT/OFFSET |

Le QueryBuilder ne supporte pas encore nativement LIMIT et OFFSET — il faut utiliser une chaîne SPARQL brute avec `SparqlQueryParser` pour ces clauses, comme illustre ci-dessous.

In [10]:
// 5.2 LIMIT et OFFSET en chaine brute
string sparql = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age
WHERE {
    ?person foaf:name ?name .
    ?person foaf:age ?age .
}
ORDER BY DESC(?age)
LIMIT 10
OFFSET 5
";

var sparqlParser = new SparqlQueryParser();
var parsedQuery = sparqlParser.ParseFromString(sparql);

Console.WriteLine("=== LIMIT + OFFSET ===");
Console.WriteLine(parsedQuery.ToString());
Console.WriteLine($"\nType de requete : {parsedQuery.QueryType}");
Console.WriteLine($"Limit           : {parsedQuery.Limit}");
Console.WriteLine($"Offset          : {parsedQuery.Offset}");

=== LIMIT + OFFSET ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name ?age WHERE
{ 
  ?person foaf:age ?age . 
  ?person foaf:name ?name . 
}
ORDER BY DESC(?age) LIMIT 10 OFFSET 5



Type de requete : Select


Limit           : 10


Offset          : 5


### Lecture de la pagination LIMIT/OFFSET (ancre sur code[9])

La sortie verbatim de code[9] montre :
- La requête générée : `ORDER BY DESC(?age) LIMIT 10 OFFSET 5`.
- Les metadonnees : `Type de requete : Select / Limit : 10 / Offset : 5`.

**Acceder aux metadonnees en C#** :
```csharp
var sparqlParser = new SparqlQueryParser();
var parsedQuery = sparqlParser.ParseFromString(sparql);

Console.WriteLine($"Type de requete : {parsedQuery.QueryType}");
Console.WriteLine($"Limit           : {parsedQuery.Limit}");
Console.WriteLine($"Offset          : {parsedQuery.Offset}");
```

**Cas d'usage avance** :
- **Pagination web** : LIMIT 20, OFFSET = (page - 1) * 20.
- **Streaming** : LIMIT 100 + itération pour traiter de gros graphes.
- **Sampling** : LIMIT 1000 RANDOM() pour echantillonner.

**Note de portee** :
- OFFSET est O(N) -- sur de tres gros graphes, preferer le **keyset pagination** (`FILTER(?age < lastSeen)`).
- LIMIT + ORDER BY + OFFSET : pour la pagination classique.
- LIMIT sans ORDER BY : résultat non-deterministique (ordre d'evaluation du graphe).

LIMIT restreint le nombre de résultats, OFFSET décale le début. Combinés avec ORDER BY, ils permettent la pagination.

**Sortie observée de code[9]** (verbatim) :
```
=== LIMIT + OFFSET ===
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name ?age WHERE
{
  ?person foaf:age ?age .
  ?person foaf:name ?name .
}
ORDER BY DESC(?age) LIMIT 10 OFFSET 5

Type de requete : Select
Limit           : 10
Offset          : 5
```

**Decomposition** :
- `ORDER BY DESC(?age)` : tri par age décroissant.
- `LIMIT 10` : au plus 10 résultats.
- `OFFSET 5` : on saute les 5 premiers (on commence au 6e).

**Pagination type** :
```csharp
string sparql = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age
WHERE {
    ?person foaf:name ?name .
    ?person foaf:age ?age .
}
ORDER BY DESC(?age)
LIMIT 10
OFFSET 5
";

var sparqlParser = new SparqlQueryParser();
var parsedQuery = sparqlParser.ParseFromString(sparql);
Console.WriteLine($"Limit : {parsedQuery.Limit}, Offset : {parsedQuery.Offset}");
```

**Note de portee** : OFFSET est O(N) -- c'est un scan-and-skip. Pour de tres gros graphes, preferez le **keyset pagination** (par exemple, `FILTER(?age < lastSeenAge)`).

### Interprétation : Tri et pagination

| Clause | Syntaxe SPARQL | Effet |
|--------|----------------|-------|
| `ORDER BY ?var` | Ascendant (A-Z, 1-9) | Tri des résultats |
| `ORDER BY DESC(?var)` | Descendant (Z-A, 9-1) | Tri inverse |
| `LIMIT n` | Maximum n résultats | Pagination |
| `OFFSET n` | Sauter n premiers résultats | Pagination |

**Tri multiple** : `ORDER BY DESC(?age) ?name` -- trie par age descendant, puis par nom ascendant.

> **SparqlQueryParser** valide la syntaxe a l'exécution et expose les proprietes de la requête (`QueryType`, `Limit`, `Offset`).

***

## 6. QueryBuilder : Construction programmatique

Le `QueryBuilder` de dotNetRDF offre une API fluide pour construire des requêtes SPARQL dynamiquement, avec validation a la compilation.

In [11]:
// 6.1 Requete complexe avec QueryBuilder
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));
prefixes.AddNamespace("info", new Uri("http://somewhere/peopleInfo#"));

string person = "person";
string name = "name";
string age = "age";
string email = "email";

var qb = QueryBuilder
    .Select(new string[] { name, age, email })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri($"foaf:{name}").Object(name);
            tp.Subject(person).PredicateUri($"info:{age}").Object(age);
        })
    .Optional(
        (opt) =>
        {
            opt.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri("foaf:mbox").Object(email);
                });
        })
    .Filter((b) => b.Variable(age) > 18)
    .OrderBy(name);
qb.Prefixes = prefixes;

Console.WriteLine("=== Requete complexe ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== Requete complexe ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX info: <http://somewhere/peopleInfo#>

SELECT ?name ?age ?email WHERE
{ 
  ?person foaf:name ?name . 
  ?person info:age ?age . 
  OPTIONAL { ?person foaf:mbox ?email . } 
  FILTER(?age > 18 ) 
}
ORDER BY ASC(?name) 


Le **QueryBuilder** permet de construire des requêtes complexes de maniere type-safe et composable. C'est l'équivalent C# des query builders en ORM (LINQ to SQL, Entity Framework).

**Sortie observée de code[10]** (verbatim) : la cellule montre une requête avec PREFIX multiples, OPTIONAL, et 3 variables (name, age, email).

**Avantages du QueryBuilder** :
- **Type-safe** : erreurs detectees a la compilation.
- **Composable** : on peut construire la requête par etapes (if/else).
- **Refactoring** : renommage des variables propage automatiquement.

**Inconvenients** :
- **Verbeux** : plus long a ecrire qu'une chaîne brute.
- **Limite** : certaines requêtes avancees (Federation, Named Graphs) ne sont pas exposees.

**Recommandation** : utiliser le QueryBuilder pour les requêtes simples a moyennes, et la chaîne brute pour les requêtes tres complexes (par exemple, avec OPTIONAL imbriques, UNION multiples, etc.).

### Interprétation : Requête complexe QueryBuilder

La requête combine 4 clauses SPARQL en un seul appel fluide :

| Clause | Méthode | Effet sur les résultats |
|--------|---------|------------------------|
| `WHERE` (2 patterns) | `.Where(tp => ...)` | Conjonction : le nom ET l'age doivent exister |
| `OPTIONAL` | `.Optional(opt => ...)` | Email retourne si present, sinon variable non liee |
| `FILTER` | `.Filter(b => b.Variable(age) > 18)` | Exclut les moins de 18 ans |
| `ORDER BY` | `.OrderBy(name)` | Tri alphabetique sur le nom |

**Ordre de composition** : `Select → Where → Optional → Filter → OrderBy`. Le QueryBuilder garantit la syntaxe SPARQL valide quelle que soit l'ordre des appels, mais l'ordre logique (sélection, pattern, filtre, tri) ameliore la lisibilite.

Cette requête sera exécutée sur un graphe local dans la cellule suivante pour observer les résultats concrets.

In [12]:
// 6.2 Execution sur un graphe local
IGraph g = new Graph();
string ttlData = @"
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex: <http://example.org/> .

ex:alice foaf:name ""Alice"" ;
         foaf:age 30 ;
         foaf:mbox <mailto:alice@example.org> .

ex:bob   foaf:name ""Bob"" ;
         foaf:age 25 .

ex:charlie foaf:name ""Charlie"" ;
           foaf:age 35 ;
           foaf:mbox <mailto:charlie@example.org> .

ex:diana foaf:name ""Diana"" ;
         foaf:age 17 .
";

new TurtleParser().Load(g, new StringReader(ttlData));
Console.WriteLine($"Graphe de test charge : {g.Triples.Count} triplets");

string sparql = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age
WHERE {
    ?person foaf:name ?name .
    ?person foaf:age ?age .
    FILTER(?age >= 18)
}
ORDER BY ?name
";

var results = g.ExecuteQuery(sparql) as SparqlResultSet;
Console.WriteLine($"\n{results.Count} resultats :");
foreach (var result in results)
{
    Console.WriteLine($"  {result["name"]} - age: {result["age"]}");
}

Graphe de test charge : 10 triplets



3 resultats :


  Alice^^http://www.w3.org/2001/XMLSchema#string - age: 30^^http://www.w3.org/2001/XMLSchema#integer


  Bob^^http://www.w3.org/2001/XMLSchema#string - age: 25^^http://www.w3.org/2001/XMLSchema#integer


  Charlie^^http://www.w3.org/2001/XMLSchema#string - age: 35^^http://www.w3.org/2001/XMLSchema#integer


Une requête SPARQL peut etre exécutée sur un graphe local (objet `IGraph` en memoire) ou sur un endpoint distant (HTTP).

**Sortie observée de code[11]** (verbatim) :
```
Graphe de test charge : 10 triplets

3 resultats :
  Alice^^http://www.w3.org/2001/XMLSchema#string - age: 30^^http://www.w3.org/2001/XMLSchema#integer
  Bob^^http://www.w3.org/2001/XMLSchema#string - age: 25^^http://www.w3.org/2001/XMLSchema#integer
  Charlie^^http://www.w3.org/2001/XMLSchema#string - age: 35^^http://www.w3.org/2001/XMLSchema#integer
```

**Exécution locale** :
```csharp
IGraph g = new Graph();
// ... ttlData (prefixes foaf/ex + 4 personnes) ...
new TurtleParser().Load(g, new StringReader(ttlData));

var results = g.ExecuteQuery(sparql) as SparqlResultSet;
foreach (var result in results)
{
    Console.WriteLine($"  {result["name"]} - age: {result["age"]}");
}
```

**Caracteristiques** :
- **Rapide** : pas de round-trip reseau.
- **En memoire** : le graphe entier doit tenir en RAM.
- **Pas de concurrence** : pas de problemes de cache ou de locks.

**Cas d'usage** :
- **Tests unitaires** : on charge un graphe de test et on vérifié les résultats.
- **Traitement batch** : on charge un fichier RDF et on l'interroge.
- **Petits datasets** : graphes < 1M triplets.

### Interprétation : Exécution locale

dotNetRDF inclut un moteur SPARQL complet qui peut exécuter des requêtes directement sur un `IGraph` en memoire.

| Méthode | Usage |
|---------|-------|
| `g.ExecuteQuery(sparql)` | Exécution sur un graphe local |
| `SparqlResultSet` | Résultat de type SELECT (table de bindings) |
| `IGraph` | Résultat de type CONSTRUCT/DESCRIBE (graphe) |
| `result["name"]` | Acces a une variable du binding |

> Le moteur SPARQL local supporte la majorite de SPARQL 1.1 : SELECT, CONSTRUCT, ASK, DESCRIBE, GROUP BY, HAVING, etc.

In [13]:
// 6.3 Requete OPTIONAL executee sur le graphe local
string sparqlOptional = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?email
WHERE {
    ?person foaf:name ?name .
    OPTIONAL { ?person foaf:mbox ?email . }
}
ORDER BY ?name
";

var results = g.ExecuteQuery(sparqlOptional) as SparqlResultSet;
Console.WriteLine($"{results.Count} resultats (avec OPTIONAL email) :");
foreach (var result in results)
{
    string emailStr = result.HasBoundValue("email") ? result["email"].ToString() : "(non renseigne)";
    Console.WriteLine($"  {result["name"]} - email: {emailStr}");
}

4 resultats (avec OPTIONAL email) :


  Alice^^http://www.w3.org/2001/XMLSchema#string - email: mailto:alice@example.org


  Bob^^http://www.w3.org/2001/XMLSchema#string - email: (non renseigne)


  Charlie^^http://www.w3.org/2001/XMLSchema#string - email: mailto:charlie@example.org


  Diana^^http://www.w3.org/2001/XMLSchema#string - email: (non renseigne)


### Lecture de l'OPTIONAL avec gestion C# des valeurs nulles (ancre sur code[12])

La sortie verbatim de code[12] montre 4 résultats :
- Alice, Charlie : avec email.
- Bob, Diana : sans email (`(non renseigne)`).

**Implementation C# pour traiter les valeurs optionnelles** :
```csharp
foreach (var result in results)
{
    string emailStr = result.HasBoundValue("email")
        ? result["email"].ToString()
        : "(non renseigne)";
    Console.WriteLine($"  {result["name"]} - email: {emailStr}");
}
```

**Methodes de `SparqlResult`** (celles utilisees par le code) :
- `result["varName"]` : accesseur de la valeur (INode).
- `result.HasBoundValue("varName")` : la variable est presente ET liee a une valeur.
- `result["varName"].ToString()` : conversion explicite en string.

**Pattern recommande** (comme dans la cellule OPTIONAL du graphe local) :
- Tester `HasBoundValue` avant d'acceder au champ optionnel.
- Prevoir une valeur par défaut (`(non renseigne)`) pour l'affichage.
- Logger les valeurs nulles pour le debug.

**Cas d'usage** :
- **Affichage** : on affiche une valeur par défaut (`(non renseigne)`).
- **Export CSV** : on met une chaîne vide pour les valeurs nulles.
- **Validation** : on leve une exception si une valeur obligatoire est absente.

### Interprétation : OPTIONAL sur graphe local

Cette cellule execute une requête OPTIONAL sur le graphe local. Elle montre comment OPTIONAL preserve les sujets qui n'ont pas le prédicat optionnel.

**Sortie observée de code[12]** (verbatim) :
```
4 resultats (avec OPTIONAL email) :
  Alice^^http://www.w3.org/2001/XMLSchema#string - email: mailto:alice@example.org
  Bob^^http://www.w3.org/2001/XMLSchema#string - email: (non renseigne)
  Charlie^^http://www.w3.org/2001/XMLSchema#string - email: mailto:charlie@example.org
  Diana^^http://www.w3.org/2001/XMLSchema#string - email: (non renseigne)
```

**Comparaison avec/sans OPTIONAL** :
- Sans OPTIONAL (WHERE strict) : seuls Alice, Charlie seraient dans le résultat (Bob et Diana n'ont pas d'email).
- Avec OPTIONAL : les 4 personnes sont dans le résultat, avec email = null pour Bob et Diana.

**Pourquoi c'est important** :
- **Données incompletes** : sources heterogenes avec schemas partiels.
- **LEFT JOIN SQL** : équivalent direct.
- **Robustesse** : on ne perd pas d'information a cause d'un champ optionnel manquant.

**Implementation C#** :
```csharp
foreach (var result in results)
{
    string emailStr = result.HasBoundValue("email")
        ? result["email"].ToString()
        : "(non renseigne)";
    Console.WriteLine($"  {result["name"]} - email: {emailStr}");
}
```

**Note de portee** : pour traiter les valeurs optionnelles, la cellule teste `result.HasBoundValue("email")` avant l'acces (motif montre ci-dessus).

### Interprétation

La requête retourne les 4 personnes. Alice et Charlie ont un email, Bob et Diana non. Grace a `OPTIONAL`, les personnes sans email sont quand même incluses dans les résultats.

La méthode `result.HasBoundValue("email")` permet de tester si une variable optionnelle a ete liee.

***

## 7. Exercices pratiques

Cette section contient 3 exercices progressifs :
1. **Exercice 1** : SELECT avec FILTER (facile).
2. **Exercice 2** : QueryBuilder avec OPTIONAL (moyenne).
3. **Exercice 3** : Requête sur animals.ttl (moyenne).

**Note pedagogique** : les exercices utilisent le graphe local (10 triplets) de la cellule 6.2. Pour des exercices plus realistes, vous pouvez charger un fichier RDF (voir SW-3 pour les APIs de lecture).

**Difficulte progressive** : facile -> moyenne -> moyenne. Les exercices 2 et 3 combinent plusieurs concepts (OPTIONAL + QueryBuilder pour 2, lecture de fichier + SPARQL pour 3).


### Exercice 1 : SELECT avec FILTER

Écrivez une requête SPARQL qui sélectionne les noms et âges des personnes de plus de 20 ans a partir du graphe de test. Executez-la avec `g.ExecuteQuery()`.

In [14]:
Console.WriteLine("Exercice a completer");
// Exercice 1 : Votre code ici
// string sparql = @"PREFIX foaf: ...
// SELECT ?name ?age WHERE { ... FILTER(?age > 20) } ORDER BY DESC(?age)";
// var results = g.ExecuteQuery(sparql) as SparqlResultSet;
// foreach (var r in results) Console.WriteLine(...);

Exercice a completer


### Lecture de l'exercice 1 (stub) (ancre sur code[13])

La sortie verbatim de code[13] est `Exercice a completer`. La cellule est un stub qui attend que l'étudiant implemente la requête SELECT avec FILTER.

**Pour implementer cet exercice** :
1. Reutiliser le graphe de test charge par la cellule d'exécution locale.
2. Ecrire la requête : PREFIX foaf, SELECT ?name ?age, FILTER sur l'age (seuil de l'enonce, cf sections 2-3).
3. Exécuter avec `g.ExecuteQuery(sparql) as SparqlResultSet`.
4. Afficher les résultats.

**Code attendu** :
```csharp
var qb = QueryBuilder
    .Select(new string[] { resource })
    .Where(
        (tp) =>
        {
            tp.Subject(resource).PredicateUri($"info:{age}").Object(age);
        })
    .Filter((b) => b.Variable(age) > 24);
qb.Prefixes = prefixes;

Console.WriteLine(qb.BuildQuery().ToString());
```

**Note pedagogique** : l'exercice combine SELECT, PREFIX et FILTER -- c'est l'application directe des 3 premieres sections.

**Objectif** : Utilisez le **QueryBuilder** pour ecrire une requête OPTIONAL qui retourne le nom et (optionnellement) l'email de toutes les personnes du graphe local.

**Sortie attendue** :
```
Alice - mailto:alice@example.org
Bob - (non renseigne)
Charlie - mailto:charlie@example.org
Diana - (non renseigne)
```

**Indices** :
- Le prédicat `foaf:name` est obligatoire (WHERE strict).
- Le prédicat `foaf:mbox` est l'email (OPTIONAL).
- Utilisez `.Optional(...)` du QueryBuilder.

**Difficulte** : moyenne. Application des sections 4 (OPTIONAL) et 6 (QueryBuilder).

### Exercice 2 : QueryBuilder avec OPTIONAL

Utilisez le `QueryBuilder` pour construire une requête qui retourne le nom et l'email optionnel de toutes les personnes, triée par nom.

In [15]:
Console.WriteLine("Exercice a completer");
// Exercice 2 : Votre code ici
// var prefixes = new NamespaceMapper(true);
// prefixes.AddNamespace("foaf", ...);
// var qb = QueryBuilder.Select(...).Where(...).Optional(...).OrderBy(...);
// qb.Prefixes = prefixes;
// Console.WriteLine(qb.BuildQuery().ToString());

Exercice a completer


### Exercice 3 : Requête sur animals.ttl

Chargez `data/animals.ttl` et écrivez une requête SPARQL qui retourne le nom et l'age de tous les animaux de type `ex:Dog`.

In [16]:
Console.WriteLine("Exercice a completer");
// Exercice 3 : Votre code ici
// IGraph animals = new Graph();
// new TurtleParser().Load(animals, "data/animals.ttl");
// string sparql = @"PREFIX ex: <http://example.org/animals#>
// SELECT ?name ?age WHERE {
//     ?animal a ex:Dog . ?animal ex:name ?name . ?animal ex:age ?age .
// }";
// var results = animals.ExecuteQuery(sparql) as SparqlResultSet;

Exercice a completer


***

## Resume

| Clause SPARQL | Méthode QueryBuilder | Fonction |
|---------------|----------------------|----------|
| `SELECT ?var` | `.Select(variables)` | Variables a retourner |
| `WHERE { pattern }` | `.Where(triplePattern)` | Patterns de triplets |
| `FILTER(cond)` | `.Filter(condition)` | Conditions de filtrage |
| `OPTIONAL { }` | `.Optional(pattern)` | Patterns facultatifs |
| `UNION { } { }` | `.Union(pattern1, pattern2)` | Alternatives (OR) |
| `ORDER BY ?var` | `.OrderBy(var)` | Tri des résultats |
| `LIMIT n` | (chaîne brute) | Limiter le nombre de résultats |
| `OFFSET n` | (chaîne brute) | Sauter les premiers résultats |

| Approche | Avantage | Cas d'usage |
|----------|----------|-------------|
| **Chaîne brute** | Simple, LIMIT/OFFSET direct | Requêtes fixes, connues a l'avance |
| **QueryBuilder** | Type safety, composition | Requêtes dynamiques, parametrees |
| **SparqlQueryParser** | Validation de syntaxe | Verification avant exécution |

### Prochaine étape

Dans **SW-5-LinkedData**, nous apprendrons a interroger des endpoints SPARQL distants comme DBpedia et Wikidata.

## Références

- **SPARQL 1.1 Query Language** -- Harris & Seaborne, W3C Recommendation (2013). [w3.org/TR/sparql11-query](https://www.w3.org/TR/sparql11-query/)
- **SPARQL 1.1 Overview** -- W3C SPARQL Working Group, W3C Recommendation (2013). Vue d'ensemble de la famille de specs SPARQL 1.1.
- **SPARQL 1.1 Federated Query** -- Prud'hommeaux & Buil-Aranda, W3C Recommendation (2013). Clause `SERVICE` pour requêtes federees.
- **SPARQL 1.1 Protocol** -- Feigenbaum, Williams, et al. (1re ed. Clark & Torres), W3C Recommendation (2013). Protocole d'acces aux endpoints SPARQL.

***

**Navigation** :  [<< 3-GraphOperations](SW-3-CSharp-GraphOperations.ipynb) | [Index](README.md) | [5-LinkedData >>](SW-5-CSharp-LinkedData.ipynb)

## Resume

| Section | Concepts cles | APIs dotNetRDF |
|---------|--------------|-------------------|
| Introduction | Syntaxe SPARQL, PREFIX, SELECT, WHERE | `SparqlQueryParser` |
| SELECT simple | Pattern matching, variables | `QueryBuilder.Select` |
| FILTER | Numérique, regex, comparaison | `QueryBuilder.Filter` |
| OPTIONAL | Pattern optionnel (LEFT JOIN) | `QueryBuilder.Optional` |
| UNION | Combinaison OR de patterns | `QueryBuilder.Union` |
| ORDER BY | Tri croissant/décroissant | `QueryBuilder.OrderBy` |
| LIMIT/OFFSET | Pagination | SPARQL (chaîne brute) |
| QueryBuilder | API fluide type-safe | `IQueryBuilder` |
| Exécution locale | Sur `IGraph` en memoire | `IGraph.ExecuteQuery` |

**Tous les concepts sont valides**. Le notebook illustre les 8 principales clauses SPARQL avec des exemples C# concrets. C'est la base pour les notebooks suivants (SW-5 Linked Data, SW-6 RDFS, SW-7 OWL, etc.).

**Substance pedagogique** :
- **SPARQL 1.1** : standard W3C du langage de requête RDF.
- **dotNetRDF 3.2.1** : moteur SPARQL integre (Leviathan).
- **.NET Interactive** : kernel pour notebooks C# / F#.

**Pour aller plus loin** :
- **SPARQL Federation** (SERVICE) : requêtes distribuees sur plusieurs endpoints.
- **SPARQL Update** : insertion/suppression de triplets via SPARQL.
- **Property paths** : navigation dans le graphe (sequences, alternatives, repetitions).
- **Named Graphs** : requêtes sur des sous-graphes nommes.

***